# ASG-Net: Adaptive Spectral-Spatial Gated Fusion Network for RGB-Multispectral Semantic Segmentation

**Paper Implementation & Reproduction Notebook**  
Author / Research Team: College Research Project  
Target Dataset: ISPRS Potsdam 2D Semantic Labeling Benchmark  
Target Application: High-Resolution Campus & Urban Semantic Mapping  

---
### Quick Setup Note for Google Colab:
1. Go to **Runtime** > **Change runtime type**.
2. Hardware accelerator: Select **T4 GPU**.
3. Click **Save**.


## 1. Environment Setup & Project Files Loading
**Step 1:** In Google Colab, click the **Folder icon 📁** on the left sidebar.
**Step 2:** Drag and drop `Dl_project.zip` into the files area (or upload it).
**Step 3:** Run the cells below to unzip the project and install all dependencies.

In [ ]:
# Verify GPU access
!nvidia-smi


In [ ]:
import os, sys, zipfile

# Automatically unzip Dl_project.zip if uploaded
zip_path = 'Dl_project.zip' if os.path.exists('Dl_project.zip') else '/content/Dl_project.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print('Successfully extracted project files from Dl_project.zip!')
else:
    print('Note: If you have not uploaded Dl_project.zip yet, drag and drop it into the left files panel.')

# Add current working directory to sys.path so 'import models' works everywhere
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)
print('Current directory:', cwd)
print('Found folders:', [d for d in os.listdir('.') if os.path.isdir(d) and not d.startswith('.')])

# Install required dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers timm albumentations opencv-python matplotlib pyyaml tqdm seaborn scikit-learn
print('Dependencies installed successfully!')


## 2. Dataset Setup: Synthetic Mini-Benchmark or Real ISPRS Potsdam
To test the full training and evaluation pipeline immediately without waiting to download the 50GB ISPRS Potsdam dataset, we provide a **Fast Synthetic Mini-Benchmark Generator** below.

*Note*: For the real ISPRS Potsdam dataset, follow the instructions in the markdown cell below.

### Real ISPRS Potsdam Download Instructions
To download the official benchmark:
1. Request access from the ISPRS 2D Semantic Labeling Contest (Potsdam dataset).
2. Download `2_Ortho_RGB.zip`, `3_Ortho_IRRG.zip`, and `5_Labels_all.zip`.
3. Run the preprocessing script:
```bash
!python data/download_potsdam.py --raw_dir /path/to/extracted_potsdam --output_dir ./data/potsdam_processed
```

In [ ]:
# Generate Synthetic Mini-Benchmark for quick testing
import numpy as np
import cv2
from pathlib import Path

mini_dir = Path('./data/potsdam_mini')
for split in ['train', 'val', 'test']:
    (mini_dir / 'rgb' / split).mkdir(parents=True, exist_ok=True)
    (mini_dir / 'nir' / split).mkdir(parents=True, exist_ok=True)
    (mini_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

num_samples = {'train': 16, 'val': 4, 'test': 4}

print('Generating synthetic multimodal patches for immediate testing...')
for split, count in num_samples.items():
    for idx in range(count):
        # Synthetic 512x512 urban patch
        rgb = np.random.randint(60, 200, (512, 512, 3), dtype=np.uint8)
        nir = np.random.randint(40, 220, (512, 512, 1), dtype=np.uint8)
        label = np.zeros((512, 512), dtype=np.uint8)  # 0: Impervious surface

        # Add synthetic building (class 1)
        bx1, by1 = np.random.randint(50, 200), np.random.randint(50, 200)
        bx2, by2 = bx1 + np.random.randint(100, 250), by1 + np.random.randint(100, 250)
        cv2.rectangle(label, (bx1, by1), (bx2, by2), 1, -1)
        cv2.rectangle(rgb, (bx1, by1), (bx2, by2), (70, 70, 180), -1)
        cv2.rectangle(nir, (bx1, by1), (bx2, by2), 50, -1)

        # Add synthetic vegetation (class 2: Low veg, class 3: Tree)
        cx, cy = np.random.randint(150, 450), np.random.randint(150, 450)
        cv2.circle(label, (cx, cy), 60, 3, -1)
        cv2.circle(rgb, (cx, cy), 60, (30, 160, 40), -1)
        cv2.circle(nir, (cx, cy), 60, 240, -1)  # High NIR reflectance for foliage!

        # Add synthetic car (class 4)
        car_x, car_y = np.random.randint(20, 100), np.random.randint(20, 100)
        cv2.rectangle(label, (car_x, car_y), (car_x + 40, car_y + 20), 4, -1)
        cv2.rectangle(rgb, (car_x, car_y), (car_x + 40, car_y + 20), (220, 220, 30), -1)

        fname = f'tile_{split}_{idx:03d}.png'
        cv2.imwrite(str(mini_dir / 'rgb' / split / fname), rgb)
        cv2.imwrite(str(mini_dir / 'nir' / split / fname), nir)
        cv2.imwrite(str(mini_dir / 'labels' / split / fname), label)

print(f'Synthetic mini-dataset successfully created at {mini_dir.resolve()}')


## 3. ASG-Net Architecture Verification
Let's instantiate ASG-Net and inspect the dual-stream MiT-B2 encoders, Cross-Modal Feature Rectification (CM-FRM), Adaptive Spectral-Spatial Gated Fusion (ASG), and boundary heads.

In [ ]:
import torch
import yaml
from models.asg_net import ASGNet

# Load config
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using compute device:', device)

# Instantiate model
model = ASGNet(config).to(device)

# Calculate parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Parameters: {total_params / 1e6:.2f} M')
print(f'Trainable Parameters: {trainable_params / 1e6:.2f} M')

# Test dummy forward pass
dummy_rgb = torch.randn(2, 3, 512, 512).to(device)
dummy_nir = torch.randn(2, 1, 512, 512).to(device)

with torch.no_grad():
    outputs = model(dummy_rgb, dummy_nir)
    seg_out = outputs['seg_pred']
    edge_out = outputs.get('edge_pred', None)

print('Segmentation Output Shape:', seg_out.shape)  # Expected: [2, 6, 128, 128] or [2, 6, 512, 512]
if edge_out is not None:
    print('Edge Output Shape:', edge_out.shape)       # Expected: [2, 1, 128, 128]
print('Forward pass completed successfully without memory errors!')


## 4. Train ASG-Net
We can now train ASG-Net using `train.py`. The training script features:
- Dual-modality synchronous data loading.
- Automatic Mixed Precision (`torch.cuda.amp.autocast`).
- Lovász-Softmax + Cross-Entropy + Auxiliary Edge Loss.
- Polynomial Learning Rate Scheduling with linear warmup.

In [ ]:
# Run training on mini-benchmark (or change --config to use full Potsdam path)
!python train.py --config configs/config.yaml \
                 --epochs 5 \
                 --batch_size 4 \
                 --lr 6e-5


## 5. Quantitative Model Evaluation
Run comprehensive evaluation computing:
- Mean Intersection-over-Union (mIoU)
- Overall Accuracy (OA)
- Macro F1 Score
- Per-class IoU breakdown & Confusion Matrix

In [ ]:
# Evaluate on the test split
!python evaluate.py --config configs/config.yaml \
                    --checkpoint checkpoints/best_model.pth \
                    --split test \
                    --save_predictions


## 6. Qualitative Prediction & Visualization
Let's visualize RGB input, NIR band, ground truth mask, and predicted segmentation map side-by-side.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import glob
from utils.visualize import colorize_mask, CLASS_COLORS, CLASS_NAMES

# Pick a sample from results or test set
test_rgbs = glob.glob('data/potsdam_mini/rgb/test/*.png')
if len(test_rgbs) > 0:
    sample_path = test_rgbs[0]
    nir_path = sample_path.replace('rgb', 'nir')
    label_path = sample_path.replace('rgb', 'labels')

    rgb = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
    nir = cv2.imread(nir_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb)
    axes[0].set_title('RGB Aerial Image')
    axes[0].axis('off')

    axes[1].imshow(nir, cmap='viridis')
    axes[1].set_title('NIR Channel (Near-Infrared)')
    axes[1].axis('off')

    gt_color = colorize_mask(gt, CLASS_COLORS)
    axes[2].imshow(gt_color)
    axes[2].set_title('Ground Truth Labels')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No test images found to display.')


## 7. Campus Drone / Custom Aerial Image Inference
To run segmentation on your college campus aerial or drone imagery:
1. If you have an RGB image + NIR band: supply both paths.
2. If you only have RGB imagery: `predict.py` automatically synthesizes an estimated NIR proxy channel.
```bash
!python predict.py --checkpoint checkpoints/best_model.pth \
                   --rgb_path path/to/campus_drone.jpg \
                   --output_dir results/campus_inference
```

In [ ]:
# Run inference on a sample image
if len(test_rgbs) > 0:
    !python predict.py --checkpoint checkpoints/best_model.pth \
                       --rgb_path {sample_path} \
                       --nir_path {nir_path} \
                       --output_dir results/sample_inference
    print('Inference complete! Output saved to results/sample_inference.')


## 8. Ablation Study Runner (Paper Benchmarking)
To reproduce the 5 ablation variants presented in the research paper:
1. `rgb_only` (Single-modality RGB baseline)
2. `early_fusion` (Early channel concatenation)
3. `dual_add` (Dual-branch element-wise addition)
4. `dual_asg_no_cmfrm` (ASG without Feature Rectification)
5. `full_asg_net` (Full ASG-Net with CM-FRM and Edge Supervision)

Run the ablation runner:

In [ ]:
# Run ablation study
# !python ablation.py --config configs/config.yaml --epochs 5 --output_dir results/ablation
print('Uncomment the line above to run the automated multi-variant ablation study.')
